In [2]:
import argparse
import dask.array as da
import dask_image.imread
import tifffile
from loguru import logger

In [3]:
import tifffile
import ome_types
from loguru import logger
from typing import Dict, Any

def read_metadata(file_path: str) -> Dict[str, Any]:
    """
    Reads metadata from a TIFF file, handling cases with no or partial metadata.

    Args:
        file_path: The absolute path to the image file.

    Returns:
        A dictionary containing extracted metadata:
        - 'pixel_size': Physical size of a pixel in the X dimension (float, defaults to 1.0).
        - 'channel_names': A list of channel names (list of str, defaults to []).
        - 'ome_metadata': The full OME metadata object (ome_types.OME, or None).
    """
    metadata = {
        'pixel_size': 1.0,
        'channel_names': [],
        'ome_metadata': None
    }

    try:
        logger.debug(f"Attempting to read metadata from: {file_path}")

        if not (file_path.lower().endswith('.tif') or file_path.lower().endswith('.tiff')):
            logger.warning(f"File does not have a .tif or .tiff extension: {file_path}")

        with tifffile.TiffFile(file_path) as tif:
            if tif.ome_metadata:
                ome = ome_types.from_xml(tif.ome_metadata)
                metadata['ome_metadata'] = ome
            else:
                logger.warning(f"No OME-XML metadata found in TIFF: {file_path}")
                logger.warning(f"Looking for pixel size in tags")
                page = tif.pages[0]
                if 'XResolution' in page.tags:
                    x_res_val = page.tags['XResolution'].value
                    metadata['pixel_size'] = x_res_val[1] / x_res_val[0]
                    logger.info(f"Found basic TIFF resolution tag: {metadata['pixel_size']:.4f}")
                return metadata

        # If we have an OME object, extract detailed information
        if metadata['ome_metadata']:
            ome = metadata['ome_metadata']
            image_meta = ome.images[0]

            # Safely extract pixel size
            if image_meta.pixels.physical_size_x is not None:
                metadata['pixel_size'] = image_meta.pixels.physical_size_x
                logger.info(f"Extracted pixel size: {metadata['pixel_size']:.4f} {image_meta.pixels.physical_size_x_unit}")
            else:
                logger.warning("PhysicalSizeX not found in OME metadata.")

            # Safely extract channel names
            if image_meta.pixels.channels:
                metadata['channel_names'] = [ch.name for ch in image_meta.pixels.channels if ch.name]
                if metadata['channel_names']:
                    logger.info(f"Extracted channel names: {metadata['channel_names']}")
                else:
                    logger.warning("Channels found, but they have no names in OME metadata.")
            else:
                logger.warning("No channels found in OME metadata.")

    except FileNotFoundError:
        logger.error(f"File not found: {file_path}")
    except Exception as e:
        logger.error(f"An unexpected error occurred while reading {file_path}: {e}")

    return metadata

In [4]:
input_path = "/Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_15_TNBC_subset.ome.tif"

In [5]:
img = dask_image.imread.imread(input_path)

In [6]:
img = img.rechunk(chunks=[1,1000,1000])

In [7]:
img.shape

(15, 5000, 5000)

In [8]:
img.chunksize

(1, 1000, 1000)

In [9]:
pyramid = [img]

while min(pyramid[-1].shape[1:]) > 256:
    downsample_factors = {1: 3, 2: 3}
    downsampled = da.coarsen(da.mean, pyramid[-1], downsample_factors, trim_excess=True).astype(img.dtype)
    pyramid.append(downsampled)

In [10]:
pyramid

[dask.array<rechunk-merge, shape=(15, 5000, 5000), dtype=uint8, chunksize=(1, 1000, 1000), chunktype=numpy.ndarray>,
 dask.array<astype, shape=(15, 1666, 1666), dtype=uint8, chunksize=(1, 334, 334), chunktype=numpy.ndarray>,
 dask.array<astype, shape=(15, 555, 555), dtype=uint8, chunksize=(1, 111, 111), chunktype=numpy.ndarray>,
 dask.array<astype, shape=(15, 185, 185), dtype=uint8, chunksize=(1, 37, 37), chunktype=numpy.ndarray>]

In [12]:
metadata = read_metadata(input_path)

2025-08-10 20:15:19.231 | DEBUG    | __main__:read_metadata:26 - Attempting to read metadata from: /Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_15_TNBC_subset.ome.tif
2025-08-10 20:15:19.240 | INFO     | __main__:read_metadata:53 - Extracted pixel size: 1.0000 UnitsLength.MICROMETER
2025-08-10 20:15:19.241 | INFO     | __main__:read_metadata:61 - Extracted channel names: ['Channel 1', 'Channel 2', 'Channel 3', 'Channel 4', 'Channel 5', 'Channel 6', 'Channel 7', 'Channel 8', 'Channel 9', 'Channel 10', 'Channel 11', 'Channel 12', 'Channel 13', 'Channel 14', 'Channel 15']


In [13]:
tifffile.imwrite(
        output_path,
        pyramid,
        bigtiff=True,
        photometric=metadata.get('photometric', 'minisblack'),
        compression=metadata.get('compression', 'zlib'),
        metadata=metadata,
        subifds=len(pyramid) - 1
    )

TypeError: imwrite() got an unexpected keyword argument 'subifds'

In [15]:
from tifffile import TiffWriter

In [26]:
metadata = read_metadata(input_path)
# ome_obj = metadata["ome_metadata"]
# ome_xml_bytes = ome_types.to_xml(ome_obj).encode('utf-8')

2025-08-10 20:34:47.688 | DEBUG    | __main__:read_metadata:26 - Attempting to read metadata from: /Users/jnimoca/Jose_BI/1_Repositories/pyimage/test_data/inputs/TD_15_TNBC_subset.ome.tif
2025-08-10 20:34:47.702 | INFO     | __main__:read_metadata:53 - Extracted pixel size: 1.0000 UnitsLength.MICROMETER
2025-08-10 20:34:47.702 | INFO     | __main__:read_metadata:61 - Extracted channel names: ['Channel 1', 'Channel 2', 'Channel 3', 'Channel 4', 'Channel 5', 'Channel 6', 'Channel 7', 'Channel 8', 'Channel 9', 'Channel 10', 'Channel 11', 'Channel 12', 'Channel 13', 'Channel 14', 'Channel 15']


In [29]:
metadata

{'pixel_size': 1.0,
 'channel_names': ['Channel 1',
  'Channel 2',
  'Channel 3',
  'Channel 4',
  'Channel 5',
  'Channel 6',
  'Channel 7',
  'Channel 8',
  'Channel 9',
  'Channel 10',
  'Channel 11',
  'Channel 12',
  'Channel 13',
  'Channel 14',
  'Channel 15'],
 'ome_metadata': OME(
    images=[<1 field_type>],
    structured_annotations={'map_annotations': [{'id': 'Annotation:Resolution:0', 'namespace': 'openmicroscopy.org/PyramidResolution', 'value': {}, 'kind': 'mapannotation'}]},
    uuid='urn:uuid:b3827596-3c3e-48f2-a9af-a44d1183d245',
    creator='OME Bio-Formats 7.0.1',
 )}

In [31]:
ome_obj = metadata['ome_metadata']

In [35]:
from ome_types.model.simple_types import UnitsLength

# Assuming ome_obj is your ome_types.OME object
pixels = ome_obj.images[0].pixels
pixels.physical_size_x = 2.0
pixels.physical_size_x_unit = UnitsLength.MICROMETER
pixels.physical_size_y = 2.0
pixels.physical_size_y_unit = UnitsLength.MICROMETER

In [36]:
pixels

Pixels(
   id='Pixels:0',
   channels=[<15 field_type>],
   tiff_data_blocks=[<15 field_type>],
   dimension_order='XYCZT',
   type='uint8',
   interleaved=False,
   big_endian=True,
   size_x=5000,
   size_y=5000,
   size_z=1,
   size_c=15,
   size_t=1,
   physical_size_x=2.0,
   physical_size_y=2.0,
)

In [37]:
ome_obj

OME(
   images=[<1 field_type>],
   structured_annotations={'map_annotations': [{'id': 'Annotation:Resolution:0', 'namespace': 'openmicroscopy.org/PyramidResolution', 'value': {}, 'kind': 'mapannotation'}]},
   uuid='urn:uuid:b3827596-3c3e-48f2-a9af-a44d1183d245',
   creator='OME Bio-Formats 7.0.1',
)

In [38]:
metadata_dict = {"axes": "CYX", "OME": ome_obj.to_xml()}

In [39]:
metadata_dict

{'axes': 'CYX',
 'OME': '<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd" UUID="urn:uuid:b3827596-3c3e-48f2-a9af-a44d1183d245" Creator="OME Bio-Formats 7.0.1">\n  <Image ID="Image:0">\n    <Pixels ID="Pixels:0" DimensionOrder="XYCZT" Type="uint8" Interleaved="false" BigEndian="true" SizeX="5000" SizeY="5000" SizeZ="1" SizeC="15" SizeT="1" PhysicalSizeX="2.0" PhysicalSizeXUnit="µm" PhysicalSizeY="2.0" PhysicalSizeYUnit="µm">\n      <Channel ID="Channel:0:0" Name="Channel 1" SamplesPerPixel="1" Color="-16777216">\n        <LightPath/>\n      </Channel>\n      <Channel ID="Channel:0:1" Name="Channel 2" SamplesPerPixel="1" Color="16711680">\n        <LightPath/>\n      </Channel>\n      <Channel ID="Channel:0:2" Name="Channel 3" SamplesPerPixel="1" Color="65280">\n        <LightPath/>\n      </Ch

In [ ]:
output_path = "./outv3_.ome.tif"

In [40]:
with TiffWriter(output_path, bigtiff=True, ome=True) as tif:
    for i, level in enumerate(pyramid):
        tif.write(
            level,
            subfiletype=1 if i > 0 else 0,
            subifds=(len(pyramid) - 1) if i == 0 else None,
            photometric="minisblack",
            compression="zlib",
            metadata=metadata_dict if i == 0 else None,
        )

TypeError: string indices must be integers, not 'str'

In [23]:
from ome_types import from_xml
from tifffile import imread, imwrite

In [24]:
with tifffile.TiffFile(input_path) as tif:
    ome_xml = tif.ome_metadata  # XML string
    ome_obj = from_xml(ome_xml)  # Convert to ome_types.OME object

In [25]:
ome_obj

OME(
   images=[<1 field_type>],
   structured_annotations={'map_annotations': [{'id': 'Annotation:Resolution:0', 'namespace': 'openmicroscopy.org/PyramidResolution', 'value': {}, 'kind': 'mapannotation'}]},
   uuid='urn:uuid:b3827596-3c3e-48f2-a9af-a44d1183d245',
   creator='OME Bio-Formats 7.0.1',
)

In [ ]:
metadata_dict = {"axes": "CYX", "OME": ome_obj.to_xml()}